## Example 1: Client initialization and connection

In [ ]:
from langgraph_sdk import get_client
import os

# Example 1: Client initialization and connection
# Mock environment setup for demo
os.environ["LANGGRAPH_API_URL"] = "http://localhost:8000"

# Initialize synchronous client
try:
    client = get_client(
        url=os.environ.get("LANGGRAPH_API_URL", "http://localhost:8000"),
        api_key=os.environ.get("LANGGRAPH_API_KEY", "demo-key")
    )
    
    # Client configuration
    print("LangGraph SDK Client initialized:")
    # The transport is exposed as `client.http`.
    print(f"  URL: {client.http.client.base_url}")
    print(f"  Headers: {len(client.http.client.headers)} configured")
    print("  Ready for: assistants, threads, runs, crons, stores")
    
except Exception as e:
    raise AssertionError(
        f"langgraph_sdk example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Assistant management operations

In [ ]:
from langgraph_sdk.schema import Assistant, AssistantVersion
from datetime import datetime

# Example 2: Assistant management operations
class AssistantManager:
    """Manage LangGraph assistants"""
    
    def create_assistant(self, graph_id: str, config: dict) -> dict:
        """Create a new assistant"""
        return {
            "assistant_id": f"asst_{datetime.now().timestamp():.0f}",
            "graph_id": graph_id,
            "config": config,
            "created_at": datetime.now().isoformat()
        }
    
    def list_versions(self, assistant_id: str) -> list:
        """List assistant versions"""
        return [
            {"version": 1, "created": "2024-01-01", "status": "deployed"},
            {"version": 2, "created": "2024-01-02", "status": "current"}
        ]

# Demo assistant management
manager = AssistantManager()
assistant = manager.create_assistant("graph_123", {"model": "gpt-4"})
versions = manager.list_versions(assistant["assistant_id"])

print(f"Assistant created: {assistant['assistant_id']}")
print(f"Graph ID: {assistant['graph_id']}")
print(f"Versions available: {len(versions)}")

## Example 3: Thread and run operations

In [ ]:
from langgraph_sdk.schema import Thread, Run
import uuid

# Example 3: Thread and run operations
class ThreadRunner:
    """Handle thread and run operations"""
    
    def create_thread(self, metadata: dict = None) -> dict:
        """Create a new conversation thread"""
        return {
            "thread_id": str(uuid.uuid4()),
            "created_at": "2024-01-01T00:00:00Z",
            "metadata": metadata or {},
            "status": "active"
        }
    
    def create_run(self, thread_id: str, assistant_id: str, input_data: dict) -> dict:
        """Create and execute a run"""
        return {
            "run_id": str(uuid.uuid4()),
            "thread_id": thread_id,
            "assistant_id": assistant_id,
            "status": "completed",
            "input": input_data,
            "output": {"response": "Processed successfully"}
        }

# Demo thread and run operations
runner = ThreadRunner()
thread = runner.create_thread({"user": "demo_user"})
run = runner.create_run(thread["thread_id"], "asst_123", {"query": "Hello"})

print(f"Thread: {thread['thread_id'][:8]}...")
print(f"Run: {run['run_id'][:8]}...")
print(f"Status: {run['status']}, Output: {run['output']}")

## Example 4: Cron job management for scheduled tasks

In [ ]:
from langgraph_sdk.schema import Cron
from datetime import datetime, timedelta

# Example 4: Cron job management for scheduled tasks
class CronManager:
    """Manage scheduled cron jobs"""
    
    def create_cron(self, assistant_id: str, schedule: str, input_data: dict) -> dict:
        """Create a scheduled cron job"""
        return {
            "cron_id": f"cron_{datetime.now().timestamp():.0f}",
            "assistant_id": assistant_id,
            "schedule": schedule,  # e.g., "0 9 * * *" for daily at 9am
            "input": input_data,
            "created_at": datetime.now().isoformat(),
            "next_run": (datetime.now() + timedelta(hours=1)).isoformat()
        }
    
    def list_crons(self, limit: int = 10) -> list:
        """List active cron jobs"""
        return [
            {"cron_id": "cron_1", "schedule": "0 * * * *", "status": "active"},
            {"cron_id": "cron_2", "schedule": "0 0 * * *", "status": "active"}
        ]

# Demo cron management
cron_mgr = CronManager()
cron = cron_mgr.create_cron("asst_123", "0 9 * * *", {"task": "daily_report"})
active_crons = cron_mgr.list_crons()

print(f"Cron created: {cron['cron_id']}")
print(f"Schedule: {cron['schedule']} (daily at 9am)")
print(f"Active crons: {len(active_crons)}")

## Example 5: Store operations for key-value storage

In [ ]:
from typing import Dict, Any, List

# Example 5: Store operations for key-value storage
class StoreManager:
    """Manage key-value store operations"""
    
    def __init__(self):
        self.store = {}
    
    def put_item(self, namespace: str, key: str, value: Any) -> dict:
        """Store an item in namespace"""
        if namespace not in self.store:
            self.store[namespace] = {}
        self.store[namespace][key] = value
        return {"namespace": namespace, "key": key, "status": "stored"}
    
    def get_item(self, namespace: str, key: str) -> Any:
        """Retrieve item from store"""
        return self.store.get(namespace, {}).get(key, None)
    
    def list_namespaces(self) -> List[str]:
        """List all namespaces"""
        return list(self.store.keys())

# Demo store operations
store = StoreManager()
store.put_item("configs", "model_settings", {"temperature": 0.7})
store.put_item("cache", "result_123", {"data": "processed"})
value = store.get_item("configs", "model_settings")

print(f"Namespaces: {store.list_namespaces()}")
print(f"Retrieved: {value}")
print(f"Items stored: {sum(len(v) for v in store.store.values())}")